# Diffusion Model Basics: Amortized SBI for a Lévy Flight Model

In this exercise we will use **BayesFlow**'s `DiffusionModel` inference network to perform
simulation-based inference (SBI) on a **Lévy flight** decision-making model — a generalization
of the classic drift-diffusion model (DDM) that cannot be fit with standard likelihood-based
methods.

By the end of this notebook you will be able to:

- Explain what a Lévy flight model is and how it differs from a standard (Gaussian-noise)
  diffusion model.
- Explain why the likelihood of the Lévy flight model is intractable, and why this makes it a
  natural candidate for simulation-based inference.
- Run a `BasicWorkflow` with a diffusion model as the inference network and a
  permutation-invariant summary network.
- Evaluate the quality of the resulting amortized posterior using standard SBI diagnostics.


In [8]:
import numpy as np
import bayesflow as bf

from ssms.basic_simulators.simulator import simulator as ssm_simulator
from ssms.config import model_config as ssms_model_config

## Background: what is a Lévy flight model?

The standard drift-diffusion model (DDM) assumes that noisy evidence accumulates over time as a
**Wiener process**: at every instant, an infinitesimally small increment of evidence is added,
drawn from a **Gaussian** distribution. This produces continuous, "wiggly" accumulation paths and
gives rise to well-known, well-behaved (inverse-Gaussian-shaped) RT distributions.

A **Lévy flight** model generalizes this idea by replacing the Gaussian noise increments with
increments drawn from an **α-stable distribution**. The extra parameter, the *stability parameter*
$\alpha \in (0, 2]$, controls how heavy-tailed these increments are:

- When $\alpha = 2$, the α-stable distribution reduces to the Gaussian, and the Lévy flight model
  reduces exactly to the standard Wiener diffusion model.
- When $\alpha < 2$, the increments become **heavy-tailed**: most steps are small, but occasional
  very large jumps occur. Sample paths of the evidence-accumulation process are no longer smooth —
  they can jump discontinuously toward (or even across) a decision boundary.

Intuitively, $\alpha$ lets us capture decision processes that are punctuated by sudden, large
shifts in evidence (e.g., abrupt attentional shifts or "bursts" of information), which a purely
Gaussian-noise diffusion model cannot represent.

In the simulator below, the Lévy flight process is parameterized by:

- `v`: drift rate (average rate of evidence accumulation)
- `a`: boundary separation (decision threshold)
- `z`: starting point / relative bias
- `alpha`: stability parameter of the noise (α = 2 => Gaussian/Wiener noise; α < 2 => heavy-tailed jumps)
- `t`: non-decision time

### Why is the likelihood intractable?

For the standard Wiener diffusion model ($\alpha = 2$), the first-passage-time distribution (i.e.,
the RT/choice likelihood) has a known closed-form (or rapidly converging series) solution. This is
what makes classic DDM fitting via maximum likelihood possible.

For $\alpha < 2$, however, the evidence-accumulation process is no longer a simple Wiener process:
it is a jump process governed by a **fractional** (rather than ordinary) Fokker–Planck equation.
There is generally **no closed-form expression** for its first-passage-time density, and evaluating the likelihood would require solving this fractional PDE numerically for every parameter setting — which is expensive and often numerically unstable. In practice, the only thing we can efficiently do is *simulate* trajectories from the model, not *evaluate* their likelihood.

This is exactly the situation SBI methods are designed for. Instead of writing down a likelihood,
we train a neural network on many simulated (parameters, data) pairs to approximate the posterior
directly, using only the simulator as our model of the world.

## Generate simulation data

The next few cells define the Lévy flight simulator and generate the training and validation data
that we will use to train our amortized inference workflow.


In [9]:
config = ssms_model_config["levy"]
param_names = config["params"]
lower_lca_no_bias = np.array(config["param_bounds"][0])
upper_lca_no_bias = np.array(config["param_bounds"][1])

In [26]:
def levy_prior():
    return {
        name: np.random.uniform(lo, hi)
        for name, lo, hi in zip(
            param_names, lower_lca_no_bias, upper_lca_no_bias
        )
    }


def levy_simulator(v, a, z, alpha, t, num_trials=60):
    """Simulate multiple (RT, choice) trials for LCA without starting-point bias."""
    result = ssm_simulator(
        theta={"v": v, "a": a, "z": z, "alpha": alpha, "t": t},
        model="levy",
        n_samples=num_trials,
        delta_t=0.001,
    )
    obs = np.array([result["rts"][:, 0], result["choices"][:, 0]]).T
    return {"obs": obs}

In [27]:
simulator = bf.make_simulator([levy_prior, levy_simulator])

In [28]:
train_data = simulator.sample(3000)
val_data = simulator.sample(100)

## Task: build the amortized inference workflow

Now that we have simulated `train_data` and `val_data`, your task is to assemble a
`bf.BasicWorkflow` that can learn to amortize Bayesian inference for the Lévy flight model.

A `BasicWorkflow` needs (at least) two networks:

1. A **summary network** that compresses a variable number of trial-level observations (`obs`,
   the simulated RT/choice pairs) into a fixed-size summary statistic. Since trials within a
   dataset are exchangeable (their order carries no information), this network should be
   **permutation invariant** — e.g. `bf.networks.SetTransformer` or `bf.networks.DeepSet`.
2. An **inference network** that learns to transform samples from a simple base distribution into
   samples from the posterior over `param_names`, conditioned on the learned summary statistics.
   Here we will use `bf.networks.DiffusionModel`.

**Your job:** fill in the `inference_network` and `summary_network` arguments below (look at the
BayesFlow API/docs for the available options and their arguments) so that the workflow is properly
configured, then train it on `train_data`.

### A note on the diffusion model's noise schedule and prediction target

Diffusion models are trained by learning to reverse a *forward* process that gradually destroys
the target (here, the parameters) with noise. Two design choices matter a lot for how well and
how stably this training works:

- **Noise schedule** (`noise_schedule="cosine"`): this controls *how much* noise is added at each
  diffusion time step. A **cosine schedule** shapes the signal-to-noise ratio so that it decays
  smoothly following a cosine curve, rather than linearly. It adds noise more gently near the start and end of the diffusion process and more quickly in the middle. This avoids destroying the signal too early or too late, which tends to improve sample quality and stabilize training, especially for the kinds of low-dimensional parameter vectors we sample here.
- **Prediction target** (`prediction_type="velocity"`): rather than predicting the injected noise
  directly, or the clean parameters directly, the network is trained to predict a *velocity*
  target that is a specific combination of the clean parameters and the noise. This "v-prediction"
  parameterization behaves more consistently across the whole diffusion trajectory (including near
  the very start and very end, where predicting noise or the clean parameters directly can become
  numerically ill-conditioned), which generally leads to more stable training.


In [ ]:
workflow = bf.BasicWorkflow(
    inference_network=None,  ### Your code here
    summary_network=None,  ### Your code here
    inference_variables=param_names,
    summary_variables="obs",
    standardize="all"
)


In [ ]:
# Train the workflow on the simulated training data, using the validation set for monitoring
# Use 20 epochs and a batch size of 32
history = None ### Your code here

In [ ]:
# Obtain 500 draws for each of the 100 validation sets
samples = None ### Your code here

In [ ]:
# Plot diagnostics
figs = None ### Your code here

In [ ]:
# Compute numerical diagnostics
table = None ### Your code here